In [0]:
# =============================================================================
# TRAVEL BOOKING SCD2 MERGE PROJECT - DATA QUALITY: BOOKING DATA VALIDATION
# =============================================================================
# Native PySpark implementation (no PyDeequ — incompatible with Spark 4.0)
# Checks: row count, completeness, non-negativity

from pyspark.sql import functions as F
import datetime as _dt

# =============================================================================
# PARAMETER EXTRACTION WITH DEFAULTS
# =============================================================================
try:
    arrival_date = dbutils.widgets.get("arrival_date")
except Exception:
    arrival_date = _dt.date.today().strftime("%Y-%m-%d")
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "travel_bookings"
try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "default"

# =============================================================================
# DQ RESULTS STORAGE SETUP
# =============================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.ops")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.dq_results (
  business_date DATE,
  dataset STRING,
  check_name STRING,
  status STRING,
  constraint STRING,
  message STRING,
  recorded_at TIMESTAMP
) USING DELTA
""")

# =============================================================================
# SOURCE DATA PREPARATION
# =============================================================================
src = spark.table(f"{catalog}.bronze.booking_inc").where(
    F.col("business_date") == F.to_date(F.lit(arrival_date))
)

# =============================================================================
# DATA QUALITY CHECKS (NATIVE PYSPARK)
# =============================================================================
# Single aggregation pass — efficient, no shuffles, one job
row_count = src.count()
agg = src.agg(
    F.count(F.when(F.col("customer_id").isNull(), 1)).alias("customer_id_nulls"),
    F.count(F.when(F.col("amount").isNull(), 1)).alias("amount_nulls"),
    F.count(F.when(F.col("amount") < 0, 1)).alias("amount_negative"),
    F.count(F.when(F.col("quantity") < 0, 1)).alias("quantity_negative"),
    F.count(F.when(F.col("discount") < 0, 1)).alias("discount_negative"),
).collect()[0]

# =============================================================================
# DQ RESULTS ASSEMBLY
# =============================================================================
now_iso = _dt.datetime.now().isoformat()

dq_results = [
    {
        "business_date": arrival_date,
        "dataset": "booking_inc",
        "check_name": "row_count",
        "status": "Success" if row_count > 0 else "Error",
        "constraint": "row_count > 0",
        "message": f"Row count: {row_count}",
        "recorded_at": now_iso,
    },
    {
        "business_date": arrival_date,
        "dataset": "booking_inc",
        "check_name": "customer_id_not_null",
        "status": "Success" if agg.customer_id_nulls == 0 else "Error",
        "constraint": "customer_id IS NOT NULL",
        "message": f"Nulls: {agg.customer_id_nulls}",
        "recorded_at": now_iso,
    },
    {
        "business_date": arrival_date,
        "dataset": "booking_inc",
        "check_name": "amount_not_null",
        "status": "Success" if agg.amount_nulls == 0 else "Error",
        "constraint": "amount IS NOT NULL",
        "message": f"Nulls: {agg.amount_nulls}",
        "recorded_at": now_iso,
    },
    {
        "business_date": arrival_date,
        "dataset": "booking_inc",
        "check_name": "amount_non_negative",
        "status": "Success" if agg.amount_negative == 0 else "Error",
        "constraint": "amount >= 0",
        "message": f"Negative values: {agg.amount_negative}",
        "recorded_at": now_iso,
    },
    {
        "business_date": arrival_date,
        "dataset": "booking_inc",
        "check_name": "quantity_non_negative",
        "status": "Success" if agg.quantity_negative == 0 else "Error",
        "constraint": "quantity >= 0",
        "message": f"Negative values: {agg.quantity_negative}",
        "recorded_at": now_iso,
    },
    {
        "business_date": arrival_date,
        "dataset": "booking_inc",
        "check_name": "discount_non_negative",
        "status": "Success" if agg.discount_negative == 0 else "Error",
        "constraint": "discount >= 0",
        "message": f"Negative values: {agg.discount_negative}",
        "recorded_at": now_iso,
    },
]

# Cast types so they align with the Delta table schema (avoids DELTA_FAILED_TO_MERGE_FIELDS)
dq_df = (
    spark.createDataFrame(dq_results)
    .withColumn("business_date", F.to_date(F.col("business_date")))
    .withColumn("recorded_at",   F.col("recorded_at").cast("timestamp"))
)

display(dq_df)

# =============================================================================
# DQ RESULTS LOGGING
# =============================================================================
dq_df.write.mode("append").saveAsTable(f"{catalog}.ops.dq_results")

# =============================================================================
# DQ VALIDATION AND ERROR HANDLING
# =============================================================================
if any(r["status"] == "Error" for r in dq_results):
    raise ValueError("DQ failed for bookings")

print("Booking DQ passed")